In [2]:
import os
import json
from dotenv import load_dotenv
from google import genai
from google.genai import types

# 1. Load configuration from your .env file
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")

# 2. Initialize the modern GenAI Client
# Using the stable 'v1' API version for production consistency
client = genai.Client(api_key=api_key)

# 3. Use the latest 3.1 series model for optimal performance
# Gemini 3.1 Flash-Lite is 2.5x faster than previous versions
MODEL_ID = "gemini-3.1-flash-lite-preview" 

def classify_customer_query(user_query, temperature=0):
    """
    Uses Gemini 3.1 to classify customer queries into structured JSON.
    """
    
    # Define the system instructions for the classification engine
    system_instruction = """
    You are an expert customer service routing engine. 
    Classify the incoming query into a primary and secondary category.
    
    Primary categories: Billing, Technical Support, Account Management, or General Inquiry.

    Secondary categories:
    - Billing: Unsubscribe/upgrade, Add payment method, Charge explanation, Dispute a charge.
    - Technical Support: General troubleshooting, Device compatibility, Software updates.
    - Account Management: Password reset, Update personal information, Close account, Account security.
    - General Inquiry: Product information, Pricing, Feedback, Speak to a human.

    Output MUST be valid JSON with keys: 'primary' and 'secondary'.
    """

    try:
        response = client.models.generate_content(
            model=MODEL_ID,
            config=types.GenerateContentConfig(
                system_instruction=system_instruction,
                temperature=temperature,
                # Forces the model to only output valid JSON
                response_mime_type="application/json" 
            ),
            contents=user_query
        )
        
        # Parse the JSON string into a Python dictionary
        return json.loads(response.text)
        
    except Exception as e:
        return {"error": str(e)}

# --- EXECUTION ---
if __name__ == "__main__":
    delimiter = "####"
    raw_query = "I want to delete my profile and all of my user data"
    formatted_query = f"{delimiter}{raw_query}{delimiter}"

    print(f"Processing query: {raw_query}")
    result = classify_customer_query(formatted_query)
    
    # Beautifully print the structured result
    print(json.dumps(result, indent=4))

Processing query: I want to delete my profile and all of my user data
{
    "primary": "Account Management",
    "secondary": "Close account"
}
